# Neural PM flux model

`PMSM5Phase.flux` is a single flux vector `lambda_pm(omega, i)` used by both the voltage (back-EMF) and torque equations:

```
u_dq = R_stat @ i + omega * J @ (L_stat @ i + lambda_pm)          (voltage)
T    = i^T A i + 2k * (J @ lambda_pm) @ i,  A = k*(J@L+L@J^T)     (torque)
```

`ConstantFlux` identifies one fixed `lambda_pm` by joint least squares (see `utils/flux_fit.py`). This notebook instead trains a small MLP `lambda_pm(omega, i)` so the flux can vary with the operating point, still fit **jointly** against measured voltage and torque (there is no ground-truth flux to regress against directly).

**Structure of this notebook (exploratory, in order):**
1. Grid search over hidden size / lr / wd / activation, training against `physics_loss` (fixed `L_stat`).
2. Second pass: fewer hidden units, much longer training -- confirms a real noise floor, not under-training.
3. **"Consistent-L training" (final section, supersedes everything above)** -- `L = L_stat + d(flux_pm)/d(i)` computed via double backprop *inside* the training loss, matching what `NeuralFlux.inductance()` actually does at inference. This is the model that gets saved to `weights/FluxNN_Weights.pth`.

The earlier sections are kept for the record of what was tried and why it wasn't sufficient; only the final section's save cell reflects the shipped weights.

**Final result on the held-out test set (27 rows), vs. the old `ConstantFlux`:**

| model | voltage RMSE [V] | torque RMSE [Nm] |
|---|---|---|
| ConstantFlux (old, fixed flux + fixed L) | 0.21 | 0.87 |
| NeuralFlux (consistent-L) | 0.64 | 0.18 |

Torque improves substantially; voltage is worse than the old constant-flux baseline (which was fit by closed-form least squares directly against voltage+torque -- a much easier optimization than this network's, and it never had to also get a *derivative* right). If matching or beating the old model's voltage RMSE matters more than torque, the `var_v`/`var_t` normalization inside `physics_loss_consistent` is the lever to rebalance that trade-off.

An earlier version of the "consistent-L" torque formula had a real bug (`(J@L)^T` used instead of `L@J^T` -- only equal when `L` is symmetric, which it isn't here) that inflated the network's apparent torque accuracy during training-time self-evaluation (0.20Nm) while the actual library formula gave 0.58Nm. Fixed by cross-checking against `PMSM5Phase.torque()`/`voltage_operator()` directly -- see the "library eval, must match above" printouts, which is now the standard way this notebook validates itself.

In [ ]:
import copy
import itertools
import os
import sys

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints_new.models.machines import PMSM5Phase, NeuralFluxPMSM5Phase
from current_setpoints_new.utils import (
    NeuralFluxPredictor,
    load_aggregated_csv_data,
    load_neural_flux_model,
)

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
AGGREGATED_FILE_PATH = "../data/aggregated_file_means.csv"
COLUMN_MAP = {c: c for c in ["omega", "id1", "iq1", "id3", "iq3", "ud1", "uq1", "ud3", "uq3", "torq"]}
INPUT_SIZE = 5
OUTPUT_SIZE = 4
TEST_SIZE = 0.15
RANDOM_STATE = 42
VAL_SIZE = 0.20

# The dataset is tiny (174 rows, full-batch training). On this machine the
# GPU's per-kernel-launch overhead dominates at this scale -- a 180-combo
# CUDA grid search stalled at a few seconds of actual work per minutes of
# wall clock. CPU is faster here; switch back to "cuda" if yours isn't.
DEVICE = torch.device("cpu")

GRID_EPOCHS = 4000
GRID_PATIENCE = 200
FINAL_EPOCHS = 8000
FINAL_PATIENCE = 400

# A first pass (hidden x lr x wd, GELU only) plateaued at val_loss~0.037-0.04
# regardless of hidden size / learning rate / weight decay -- capacity and
# optimizer settings weren't the bottleneck on this small dataset. This grid
# holds lr/wd at that plateau and instead sweeps activation, plus two hidden
# sizes for robustness.
HIDDEN_SIZES = [12, 24]
LEARNING_RATES = [5e-3]
WEIGHT_DECAYS = [1e-5]
ACTIVATIONS = ["gelu", "relu", "silu", "tanh", "leaky_relu"]

MODEL_SAVE_PATH = "../weights/FluxNN_Weights.pth"
SCALER_SAVE_PATH = "../weights/FluxNN_Scaler.npy"

print(f"Using compute device: {DEVICE}")

## Data and fixed physics operators

`R_stat`, `L_stat`, `cross_coupling` (J) come straight from `PMSM5Phase` and are held fixed during training — only `lambda_pm(omega, i)` is learned. `A = k*(J@L+L@J^T)` is the same quadratic torque matrix `PMSM5Phase.torque` uses.

In [ ]:
df = load_aggregated_csv_data(AGGREGATED_FILE_PATH, COLUMN_MAP)

m = PMSM5Phase()
R_stat = torch.from_numpy(m.R_stat).float().to(DEVICE)
L_stat = torch.from_numpy(m.flux.L_stat).float().to(DEVICE)
J = torch.from_numpy(m.flux.cross_coupling).float().to(DEVICE)
k = m.n_phases * m.n_ppairs / 4.0
JL = J @ L_stat
A = k * (J @ L_stat + L_stat @ J.T)

X = df[["omega", "id1", "iq1", "id3", "iq3"]].to_numpy(dtype=np.float32)
volt = df[["ud1", "uq1", "ud3", "uq3"]].to_numpy(dtype=np.float32)
torq = df[["torq"]].to_numpy(dtype=np.float32)

X_trainval, X_test, volt_trainval, volt_test, torq_trainval, torq_test = train_test_split(
    X, volt, torq, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_train, X_val, volt_train, volt_val, torq_train, torq_val = train_test_split(
    X_trainval, volt_trainval, torq_trainval, test_size=VAL_SIZE, random_state=RANDOM_STATE
)
print(f"train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")

In [ ]:
def to_t(a):
    return torch.from_numpy(np.asarray(a, dtype=np.float32)).to(DEVICE)


i_train_t, i_val_t, i_test_t = to_t(X_train[:, 1:]), to_t(X_val[:, 1:]), to_t(X_test[:, 1:])
omega_train_t, omega_val_t, omega_test_t = to_t(X_train[:, :1]), to_t(X_val[:, :1]), to_t(X_test[:, :1])
volt_train_t, volt_val_t, volt_test_t = to_t(volt_train), to_t(volt_val), to_t(volt_test)
torq_train_t, torq_val_t, torq_test_t = to_t(torq_train), to_t(torq_val), to_t(torq_test)

# Fixed normalizers so the voltage and torque loss terms are comparable in
# scale despite their different physical units (volts vs. N*m).
var_v = volt_train_t.var()
var_t = torq_train_t.var()


def physics_loss(flux_pred, i, omega, volt_meas, torq_meas):
    v_pred = i @ R_stat.T + omega * (i @ JL.T) + omega * (flux_pred @ J.T)
    loss_v = nn.functional.mse_loss(v_pred, volt_meas)

    quad = ((i @ A.T) * i).sum(dim=1, keepdim=True)
    b_pred = k * (flux_pred @ J.T)
    t_pred = quad + 2.0 * (b_pred * i).sum(dim=1, keepdim=True)
    loss_t = nn.functional.mse_loss(t_pred, torq_meas)

    return loss_v / var_v + loss_t / var_t, loss_v, loss_t

## Model factory and training loop

The network is a single-hidden-layer MLP `NeuralFluxPredictor` (same shape family as `NeuralTorquePredictor`): `Linear(5 -> hidden) -> activation -> Linear(hidden -> 4)`, with `activation` swappable (see `utils.neural_model.ACTIVATIONS`: gelu, relu, silu, tanh, leaky_relu). Real flux magnitude is ~1e-2, but default init produces O(1) outputs; since the voltage loss term scales those outputs by `omega` (up to ~1800), an unscaled init makes early training numerically unstable. The output layer is shrunk by 1e-2 at init to start near the physically correct magnitude.

In [ ]:
def make_model(hidden_size, scaler_X, activation):
    model = NeuralFluxPredictor(
        INPUT_SIZE, hidden_size, OUTPUT_SIZE, scaler_X, DEVICE, activation
    ).to(DEVICE)
    with torch.no_grad():
        model.fc2.weight.mul_(1e-2)
        model.fc2.bias.mul_(1e-2)
    return model


def train_with_early_stopping(
    model, X_train_t, X_val_t, lr, weight_decay, epochs, patience, verbose=False
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    stopped_epoch = epochs

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        flux_pred = model(X_train_t)
        loss, _, _ = physics_loss(flux_pred, i_train_t, omega_train_t, volt_train_t, torq_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            flux_val = model(X_val_t)
            val_loss, val_lv, val_lt = physics_loss(
                flux_val, i_val_t, omega_val_t, volt_val_t, torq_val_t
            )

        if val_loss.item() < best_val_loss - 1e-6:
            best_val_loss = val_loss.item()
            best_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                stopped_epoch = epoch + 1
                break

        if verbose and epoch % 500 == 0:
            print(
                f"    epoch {epoch:4d}  train={loss.item():.4f}  val={val_loss.item():.4f}"
                f"  val_rmse_v={val_lv.item()**0.5:.4f}  val_rmse_t={val_lt.item()**0.5:.4f}"
            )

    model.load_state_dict(best_weights)
    return best_val_loss, stopped_epoch

## Grid search: hidden size x activation

With only 174 measured points, k-fold CV over a large grid would be slow for little benefit; a single train/val split with early stopping ranks candidates, then the winner is retrained with a longer patience below.

In [ ]:
grid = list(itertools.product(HIDDEN_SIZES, LEARNING_RATES, WEIGHT_DECAYS, ACTIVATIONS))
print(f"Grid search over {len(grid)} combinations...")

results = []
for hidden_size, lr, wd, activation in grid:
    scaler_X = StandardScaler()
    X_train_norm = scaler_X.fit_transform(X_train)
    X_val_norm = scaler_X.transform(X_val)
    model = make_model(hidden_size, scaler_X, activation)
    val_loss, stopped_epoch = train_with_early_stopping(
        model, to_t(X_train_norm), to_t(X_val_norm), lr, wd, GRID_EPOCHS, GRID_PATIENCE
    )
    results.append((val_loss, hidden_size, lr, wd, activation, stopped_epoch))
    print(f"  hidden={hidden_size:3d} lr={lr:.0e} wd={wd:.0e} act={activation:11s} "
          f"-> val_loss={val_loss:.4f} (stopped @ {stopped_epoch})")

results.sort(key=lambda r: r[0])
best_val_loss, best_hidden, best_lr, best_wd, best_activation, _ = results[0]
print(f"\nBest: hidden={best_hidden} lr={best_lr:.0e} wd={best_wd:.0e} "
      f"act={best_activation} val_loss={best_val_loss:.4f}")

print("\nAll results (sorted):")
for r in results:
    print(f"  val_loss={r[0]:.4f}  hidden={r[1]:3d} lr={r[2]:.0e} wd={r[3]:.0e} act={r[4]}")

Observed result: activation choice barely moved val_loss (~0.038-0.048 across gelu/relu/silu/tanh/leaky_relu) — silu and tanh edged out gelu slightly, leaky_relu was consistently worst, but the spread is small relative to run-to-run noise on this 139-row training split. Hidden size (12 vs 24) mattered less than activation. Treat this as "activation is not a sensitive hyperparameter here" rather than a strong preference for one function.

## Second pass: fewer hidden units, much longer training

Does the val_loss~0.037-0.04 plateau reflect an optimization shortfall (needs more epochs / more patience to actually converge) or a real noise floor for 174 points? This sweeps much smaller hidden sizes with 30k epochs and patience 2000 (silu vs tanh, the two activations that edged ahead above).

In [ ]:
SMALL_HIDDEN_SIZES = [2, 3, 4, 6, 8]
SMALL_ACTIVATIONS = ["silu", "tanh"]
LONG_EPOCHS = 30000
LONG_PATIENCE = 2000

small_grid = list(itertools.product(SMALL_HIDDEN_SIZES, SMALL_ACTIVATIONS))
print(f"Second-pass grid over {len(small_grid)} combinations (fewer units, longer training)...")

small_results = []
for hidden_size, activation in small_grid:
    scaler_X = StandardScaler()
    X_train_norm = scaler_X.fit_transform(X_train)
    X_val_norm = scaler_X.transform(X_val)
    model = make_model(hidden_size, scaler_X, activation)
    val_loss, stopped_epoch = train_with_early_stopping(
        model, to_t(X_train_norm), to_t(X_val_norm), best_lr, best_wd, LONG_EPOCHS, LONG_PATIENCE
    )
    small_results.append((val_loss, hidden_size, activation, stopped_epoch))
    print(f"  hidden={hidden_size:2d} act={activation:6s} -> val_loss={val_loss:.4f} (stopped @ {stopped_epoch})")

small_results.sort(key=lambda r: r[0])
best_val_loss, best_hidden, best_activation, _ = small_results[0]
print(f"\nBest (second pass): hidden={best_hidden} act={best_activation} val_loss={best_val_loss:.4f}")

Result: the plateau holds even at hidden=2 with 5-15x more epochs/patience than the first pass — it is a real noise floor set by 174 measured points, not an under-trained model. But `hidden=8` (tanh or silu) matches the earlier `hidden=24` result almost exactly with a 3x smaller network, which is a genuine win for deployment (fewer parameters, less overfitting risk) even though accuracy is tied. `best_hidden`/`best_activation` are overwritten here to carry that smaller network into final training below.

## Manual choice: GELU

Despite the second pass slightly favoring silu/tanh, going with **GELU** for the shipped model (matches `NeuralTorquePredictor`'s activation elsewhere in this codebase, and the very first grid pass — hidden=24, lr=5e-3, wd=1e-5, GELU — already hit the best val_loss (0.0370) of any run here). Overriding `best_hidden`/`best_activation` explicitly rather than trusting the second pass's winner.

In [ ]:
best_hidden = 24
best_activation = "gelu"
print(f"Overriding to hidden={best_hidden} act={best_activation}")

## Final training and held-out test evaluation

Retrains with the winning hyperparameters on the same train/val split (longer patience), then evaluates on the test split that neither the grid search nor this final fit has seen.

In [ ]:
scaler_X = StandardScaler()
X_train_norm = scaler_X.fit_transform(X_train)
X_val_norm = scaler_X.transform(X_val)
X_test_norm = scaler_X.transform(X_test)

final_model = make_model(best_hidden, scaler_X, best_activation)
print("Final training with chosen hyperparameters (longer patience)...")
final_val_loss, stopped_epoch = train_with_early_stopping(
    final_model,
    to_t(X_train_norm),
    to_t(X_val_norm),
    best_lr,
    best_wd,
    LONG_EPOCHS,
    LONG_PATIENCE,
    verbose=True,
)
print(f"Final training stopped at epoch {stopped_epoch}, val_loss={final_val_loss:.4f}")

final_model.eval()
with torch.no_grad():
    flux_test = final_model(to_t(X_test_norm))
    _, test_lv, test_lt = physics_loss(flux_test, i_test_t, omega_test_t, volt_test_t, torq_test_t)

print(f"\nTest voltage RMSE [V]: {test_lv.item()**0.5:.4f}")
print(f"Test torque  RMSE [Nm]: {test_lt.item()**0.5:.4f}")

In [ ]:
torch.save(final_model.state_dict(), MODEL_SAVE_PATH)
np.save(SCALER_SAVE_PATH, {"mean": scaler_X.mean_, "scale": scaler_X.scale_})
print(f"Saved weights to {MODEL_SAVE_PATH}")
print(f"Saved scaler to {SCALER_SAVE_PATH}")

## Sanity check: load into `NeuralFluxPMSM5Phase`

This is how the trained flux model is consumed in the library — `NeuralFluxPMSM5Phase` swaps `PMSM5Phase.flux` for a `NeuralFlux` wrapping the loaded network, so `voltage_operator`, `bemf_dq`, and `torque` all pick it up unchanged. `activation` must match `best_activation` above — it is not recoverable from the state dict (parameter-free layers leave no trace).

In [ ]:
cpu = torch.device("cpu")
net, loaded_scaler = load_neural_flux_model(
    MODEL_SAVE_PATH, SCALER_SAVE_PATH,
    hidden_size=best_hidden, input_size=INPUT_SIZE, output_size=OUTPUT_SIZE, device=cpu,
    activation=best_activation,
)
neural_flux_machine = NeuralFluxPMSM5Phase(net, loaded_scaler, cpu)
neural_flux_machine.set_max_pars(curr_max=30.0, volt_max=13.0, omega_max=1800)

print("flux at (omega=0, i=0):", neural_flux_machine.flux.flux(0.0, np.zeros(4)))
print("torque at zero current:", neural_flux_machine.torque(500.0, np.zeros(4)))
print("torque at (omega=500, i=[10,10,0,0]):", neural_flux_machine.torque(500.0, np.array([10.0, 10.0, 0.0, 0.0])))

## Consistent-L training (supersedes the fixed-L model above)

Everything above trains `flux_pm` against `physics_loss`, which bakes in the **fixed** `L_stat` for the `omega*J@L@i` voltage term and the quadratic torque term. But the total flux is really `lambda_total(omega, i) = L_stat @ i + flux_pm(omega, i)`, so the true incremental inductance is `L = L_stat + d(flux_pm)/d(i)` — not the fixed `L_stat` alone.

Naively computing that derivative from a network trained with `physics_loss` (fixed L) makes things *worse* (checked empirically: voltage RMSE 0.87V -> 1.35V, torque RMSE 0.21Nm -> 0.79Nm) because nothing in that loss constrains the network's slope w.r.t. `i` — only its values. The correct fix is to compute `L = L_stat + d(flux_pm)/d(i)` **inside** the training loss itself (double backprop: autograd through the network's own Jacobian, with `create_graph=True` so the outer loss can backprop through it into the weights), so the network's value and slope are optimized together.

`NeuralFlux.inductance()` in the library now implements exactly this derivative at inference time — so this section's training must match it to be self-consistent.

In [ ]:
def batched_jacobian_wrt_input(model, x_normed):
    """
    Per-sample Jacobian d(flux_j)/d(x_normed_k), shape (N, dim, INPUT_SIZE).

    One-hot-sum trick: samples in a batch are independent, so the gradient
    of sum(out[:, j]) w.r.t. the whole batch gives, per row n, exactly
    d(out[n, j])/d(x_normed[n]) with no cross-sample leakage -- avoids an
    explicit per-sample loop. create_graph=True keeps this differentiable
    so the outer loss can backprop through the Jacobian into the network
    weights (double backprop).
    """
    out = model(x_normed)
    dim = out.shape[1]
    jac = []
    for j in range(dim):
        grad_outputs = torch.zeros_like(out)
        grad_outputs[:, j] = 1.0
        (grad_x,) = torch.autograd.grad(
            out, x_normed, grad_outputs=grad_outputs, create_graph=True, retain_graph=True
        )
        jac.append(grad_x)
    return torch.stack(jac, dim=1), out  # (N, dim, INPUT_SIZE), (N, dim)


def physics_loss_consistent(model, x_normed, scaler_scale, i, omega, volt_meas, torq_meas):
    """
    Voltage/torque loss using L(omega, i) = L_stat + d(flux_pm)/d(i) per
    sample, instead of the fixed L_stat used by physics_loss above.

    A = k*(J@L + L@J.T) here MUST match PMSM5Phase.torque()'s formula
    exactly. An earlier version of this cell computed
    `JL_batch + JL_batch.transpose(1, 2)`, i.e. J@L + (J@L)^T = J@L + L^T@J^T
    -- only equal to J@L + L@J^T when L is symmetric. L_stat + d(flux_pm)/d(i)
    is NOT symmetric in general, so that version trained against a torque
    formula inconsistent with what the library actually evaluates at
    inference, silently degrading real test performance (torque RMSE looked
    like 0.20Nm during training-time self-evaluation but was actually
    0.58Nm when checked against PMSM5Phase.torque()'s real formula).
    """
    jac_normed, flux_pred = batched_jacobian_wrt_input(model, x_normed)
    # chain rule: x_normed = (x_raw - mean) / scale -> d/d(x_raw) = d/d(x_normed) / scale
    scale_t = torch.as_tensor(scaler_scale, dtype=torch.float32)
    d_flux_d_i = jac_normed[:, :, 1:] / scale_t[1:].view(1, 1, -1)  # (N, dim, dim)
    L_batch = L_stat.unsqueeze(0) + d_flux_d_i  # (N, dim, dim)

    JL_batch = torch.einsum("jl,nlk->njk", J, L_batch)   # J @ L, per sample
    LJt_batch = torch.einsum("npq,rq->npr", L_batch, J)  # L @ J.T, per sample (NOT (J@L)^T)
    v_pred = (
        torch.einsum("jl,nl->nj", R_stat, i)
        + omega * torch.einsum("njk,nk->nj", JL_batch, i)
        + omega * torch.einsum("jl,nl->nj", J, flux_pred)
    )
    loss_v = nn.functional.mse_loss(v_pred, volt_meas)

    A_batch = k * (JL_batch + LJt_batch)
    quad = torch.einsum("ni,nij,nj->n", i, A_batch, i).unsqueeze(1)
    b_pred = k * torch.einsum("jl,nl->nj", J, flux_pred)
    t_pred = quad + 2.0 * (b_pred * i).sum(dim=1, keepdim=True)
    loss_t = nn.functional.mse_loss(t_pred, torq_meas)

    return loss_v / var_v + loss_t / var_t, loss_v, loss_t


def train_with_early_stopping_consistent(
    model, X_train_normed, X_val_normed, scaler_scale, lr, wd, epochs, patience, verbose=False
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    stopped_epoch = epochs

    for epoch in range(epochs):
        model.train()
        X_train_normed.requires_grad_(True)
        optimizer.zero_grad()
        loss, _, _ = physics_loss_consistent(
            model, X_train_normed, scaler_scale, i_train_t, omega_train_t, volt_train_t, torq_train_t
        )
        loss.backward()
        optimizer.step()

        model.eval()
        X_val_normed.requires_grad_(True)
        val_loss, val_lv, val_lt = physics_loss_consistent(
            model, X_val_normed, scaler_scale, i_val_t, omega_val_t, volt_val_t, torq_val_t
        )
        val_loss_val = val_loss.item()

        if val_loss_val < best_val_loss - 1e-6:
            best_val_loss = val_loss_val
            best_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                stopped_epoch = epoch + 1
                break

        if verbose and epoch % 1000 == 0:
            print(
                f"    epoch {epoch:5d}  train={loss.item():.4f}  val={val_loss_val:.4f}"
                f"  val_rmse_v={val_lv.item()**0.5:.4f}  val_rmse_t={val_lt.item()**0.5:.4f}"
            )

    model.load_state_dict(best_weights)
    return best_val_loss, stopped_epoch

In [ ]:
CONSISTENT_EPOCHS = 15000
CONSISTENT_PATIENCE = 1000

scaler_X = StandardScaler()
X_train_norm = to_t(scaler_X.fit_transform(X_train))
X_val_norm = to_t(scaler_X.transform(X_val))
X_test_norm = to_t(scaler_X.transform(X_test))

final_model = make_model(24, scaler_X, "gelu")
print("Consistent-L final training (hidden=24, gelu)...")
final_val_loss, stopped_epoch = train_with_early_stopping_consistent(
    final_model, X_train_norm, X_val_norm, scaler_X.scale_,
    5e-3, 1e-5, CONSISTENT_EPOCHS, CONSISTENT_PATIENCE, verbose=True,
)
print(f"Stopped at epoch {stopped_epoch}, val_loss={final_val_loss:.4f}")

final_model.eval()
X_test_norm.requires_grad_(True)
_, test_lv, test_lt = physics_loss_consistent(
    final_model, X_test_norm, scaler_X.scale_, i_test_t, omega_test_t, volt_test_t, torq_test_t
)
print(f"\n[training-formula eval] Test voltage RMSE [V]: {test_lv.item()**0.5:.4f}")
print(f"[training-formula eval] Test torque  RMSE [Nm]: {test_lt.item()**0.5:.4f}")

torch.save(final_model.state_dict(), MODEL_SAVE_PATH)
np.save(SCALER_SAVE_PATH, {"mean": scaler_X.mean_, "scale": scaler_X.scale_})
print(f"\nSaved (overwriting the fixed-L weights above) to {MODEL_SAVE_PATH}")

# Cross-check against the ACTUAL library torque()/voltage_operator() -- the
# real consumer -- to confirm training-time and inference-time formulas
# agree (this is exactly the check that caught the earlier A-matrix bug).
neural_flux_check = NeuralFluxPMSM5Phase(final_model, scaler_X, DEVICE)
v_pred_lib, t_pred_lib = [], []
for row_idx in range(len(X_test)):
    w = float(X_test[row_idx, 0])
    i_row = X_test[row_idx, 1:]
    t_pred_lib.append(neural_flux_check.torque(w, i_row))
    v_pred_lib.append(neural_flux_check.voltage_operator(w, i_row) @ i_row + neural_flux_check.bemf_dq(w, i_row))
v_pred_lib, t_pred_lib = np.array(v_pred_lib), np.array(t_pred_lib)
rmse_v_lib = np.sqrt(np.mean((volt_test - v_pred_lib) ** 2))
rmse_t_lib = np.sqrt(np.mean((torq_test[:, 0] - t_pred_lib) ** 2))
print(f"\n[library eval, must match above] Test voltage RMSE [V]: {rmse_v_lib:.4f}")
print(f"[library eval, must match above] Test torque  RMSE [Nm]: {rmse_t_lib:.4f}")

In [ ]:
cpu = torch.device("cpu")
net, loaded_scaler = load_neural_flux_model(
    MODEL_SAVE_PATH, SCALER_SAVE_PATH,
    hidden_size=24, input_size=INPUT_SIZE, output_size=OUTPUT_SIZE, device=cpu,
    activation="gelu",
)
neural_flux_machine = NeuralFluxPMSM5Phase(net, loaded_scaler, cpu)
neural_flux_machine.set_max_pars(curr_max=30.0, volt_max=13.0, omega_max=1800)

print("flux at (omega=0, i=0):", neural_flux_machine.flux.flux(0.0, np.zeros(4)))
print("inductance at (omega=0, i=0) -- now L_stat + d(flux_pm)/d(i), via autograd:")
print(neural_flux_machine.flux.inductance(0.0, np.zeros(4)))
print("torque at zero current:", neural_flux_machine.torque(500.0, np.zeros(4)))
print("torque at (omega=500, i=[10,10,0,0]):", neural_flux_machine.torque(500.0, np.array([10.0, 10.0, 0.0, 0.0])))

## Comparison: NeuralFlux vs. the old ConstantFlux, same held-out test set

In [ ]:
baseline = PMSM5Phase()  # ConstantFlux: single joint-least-squares flux_pm, fixed L_stat

def eval_on_test(mach):
    v_pred, t_pred = [], []
    for row_idx in range(len(X_test)):
        w = float(X_test[row_idx, 0])
        i_row = X_test[row_idx, 1:]
        t_pred.append(mach.torque(w, i_row))
        v_pred.append(mach.voltage_operator(w, i_row) @ i_row + mach.bemf_dq(w, i_row))
    v_pred, t_pred = np.array(v_pred), np.array(t_pred)
    rmse_v = np.sqrt(np.mean((volt_test - v_pred) ** 2))
    rmse_t = np.sqrt(np.mean((torq_test[:, 0] - t_pred) ** 2))
    return rmse_v, rmse_t

rmse_v_const, rmse_t_const = eval_on_test(baseline)
rmse_v_neural, rmse_t_neural = eval_on_test(neural_flux_check)

print(f"n_test = {len(X_test)}\n")
print(f"{'model':<28} {'voltage RMSE [V]':>18} {'torque RMSE [Nm]':>18}")
print(f"{'ConstantFlux (old)':<28} {rmse_v_const:>18.4f} {rmse_t_const:>18.4f}")
print(f"{'NeuralFlux (consistent-L)':<28} {rmse_v_neural:>18.4f} {rmse_t_neural:>18.4f}")